# Задание №7: Прогнозирование погоды с использованием LSTM сети (анализ временных рядов)

Данное задание представляет собой пошаговое руководство по созданию модели прогнозирования погоды с использованием рекуррентных нейронных сетей (LSTM). Вы познакомитесь с основными понятиями анализа временных рядов, научитесь подготавливать метеорологические данные, строить и обучать LSTM‑модель для прогнозирования температуры (или другого параметра), а также оценивать качество прогноза. Задание адаптировано для выполнения в **Google Colab** (Jupyter Notebook). Все части работы (подготовка данных, обучение, сохранение, загрузка из репозитория и прогнозирование) выполняются в одном ноутбуке.

---

## 1. Теоретическое введение: ключевые понятия анализа временных рядов

### 1.1. Временной ряд (Time Series)
Временной ряд – это последовательность наблюдений, упорядоченных по времени (например, ежедневная температура, почасовая влажность). Задача прогнозирования временного ряда заключается в предсказании будущих значений на основе исторических данных.

### 1.2. Рекуррентные нейронные сети (RNN) и LSTM
RNN – класс сетей, способных обрабатывать последовательности произвольной длины благодаря внутренней памяти. **LSTM (Long Short‑Term Memory)** – разновидность RNN, которая эффективно обучается на длинных последовательностях, избегая проблемы затухания градиента. LSTM содержит специальные «вентили», управляющие потоком информации, что позволяет запоминать зависимости на больших временных интервалах – критически важно для метеорологических данных, где сезонные и циклические закономерности могут растягиваться на недели и месяцы.

### 1.3. Окно предыстории (look‑back)
При обучении модели временных рядов данные преобразуются в обучающие пары: вход – несколько предыдущих значений (look‑back), выход – следующее значение. Размер окна выбирается исходя из предполагаемой периодичности (например, 24 часа для суточных циклов, 7 дней для недельных).

### 1.4. Масштабирование данных (Normalization)
Нейронные сети лучше обучаются, когда входные данные находятся в небольшом диапазоне (обычно [0,1] или [-1,1]). Для этого используют **MinMaxScaler** или **StandardScaler**. После прогнозирования значения возвращаются к исходному масштабу (inverse transform).

### 1.5. Функция потерь и метрики
Для задачи регрессии (прогнозирование непрерывной величины) обычно используют **среднеквадратичную ошибку (MSE)** или **среднюю абсолютную ошибку (MAE)**. Визуально качество оценивают, сравнивая графики реальных и предсказанных значений.

### 1.6. Оценка качества
Для количественной оценки применяют **RMSE (Root Mean Square Error)**, **MAE**, **MAPE** (средняя абсолютная процентная ошибка). Чем меньше эти метрики, тем точнее прогноз.

---

## 2. Постановка задачи

Вам необходимо создать собственный репозиторий на GitHub, загрузить в него файл с историческими метеорологическими данными (например, CSV с колонками: дата, температура, давление и т.д.), затем построить и обучить LSTM‑модель для прогнозирования температуры (или другого параметра) на один шаг вперёд. После обучения вы сохраните модель, скейлер и (при необходимости) другие объекты, загрузите их в тот же репозиторий. В финальной части ноутбука вы продемонстрируете, как загрузить эти файлы из репозитория и использовать их для прогнозирования новых данных без повторного обучения. Весь код и отчёт должны быть оформлены в одном Jupyter Notebook.

---

## 3. Структура отчёта

Ваш отчёт должен содержать следующие разделы (в виде ячеек Markdown в ноутбуке):

1. **Титульный лист** (название работы, ФИО, группа, ссылка на репозиторий)
2. **Введение** (цель работы, краткое описание задачи прогнозирования временных рядов)
3. **Теоретическая часть** (объяснение ключевых понятий: временной ряд, LSTM, look‑back, масштабирование, метрики)
4. **Описание данных** (источник данных, период, частота, перечень параметров; ссылка на файл в репозитории)
5. **Подготовка данных** (загрузка, обработка пропусков, визуализация, масштабирование, создание окон, разделение на train/test)
6. **Построение модели** (архитектура LSTM, визуализация модели, summary)
7. **Обучение модели** (параметры обучения, графики потерь)
8. **Прогнозирование и оценка** (прогноз на тестовой выборке, расчёт RMSE/MAE, визуализация сравнения)
9. **Функция прогнозирования** (описание функции `predict_next`, демонстрация примера)
10. **Сохранение модели и вспомогательных объектов** (код сохранения модели, скейлера; загрузка файлов в репозиторий)
11. **Загрузка модели из репозитория и быстрый прогноз** (код загрузки через raw‑ссылку, повторное использование для прогноза)
12. **Выводы** (что получилось, какие были трудности, возможные улучшения)
13. **Список использованных источников**
14. **Приложение** (полный код с комментариями)

---

## 4. Источники данных для прогнозирования погоды

### 4.1. Открытые датасеты

| Источник | Описание | Ссылка |
|----------|----------|--------|
| **Архив погоды RP5** | Исторические данные по многим городам мира (температура, давление, влажность, ветер) | https://rp5.ru/Архив_погоды |
| **NOAA Climate Data Online** | Глобальные данные от Национального управления океанических и атмосферных исследований США | https://www.ncdc.noaa.gov/cdo-web/ |
| **OpenWeatherMap** | API для получения исторических данных (требуется регистрация) | https://openweathermap.org/history |
| **Kaggle: Weather Dataset** | Популярный датасет с почасовыми метеоданными | https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data |
| **Метеостанции мира (CSV)** | Собранные данные с открытых источников | Поиск на GitHub по запросу “weather data csv” |

### 4.2. Синтетический датасет (для тренировки)

Если у вас нет возможности скачать реальные данные, вы можете сгенерировать синтетический временной ряд, имитирующий суточные и сезонные колебания, с добавлением шума. Пример:

```python
import pandas as pd
import numpy as np

# Генерация ежедневных данных за 3 года
dates = pd.date_range('2020-01-01', periods=1095, freq='D')
t = np.arange(len(dates))
# Сезонная составляющая (годовой цикл)
seasonal = 10 * np.sin(2 * np.pi * t / 365)
# Тренд (медленный рост)
trend = 0.005 * t
# Шум
noise = np.random.normal(0, 1, size=len(t))
temperature = 15 + seasonal + trend + noise

df = pd.DataFrame({'date': dates, 'temperature': temperature})
df.to_csv('weather_data.csv', index=False)
```

### 4.3. Требования к объёму данных
- **Минимальный объём**: не менее 500 точек (дней) для обучения, желательно наличие нескольких годовых циклов (≥2).
- **Максимальный объём**: в Google Colab ограничение по памяти; до 100 000 точек – комфортно.

### 4.4. Создание репозитория и загрузка данных
1. Зарегистрируйтесь на [GitHub](https://github.com).
2. Создайте новый публичный репозиторий с названием, например, `weather-forecast-lstm`.
3. Загрузите в репозиторий файл с данными (например, `weather_data.csv`).
4. Получите **raw‑ссылку** на файл (открыть файл → Raw → скопировать URL). Пример:  
   `https://raw.githubusercontent.com/ваш_логин/weather-forecast-lstm/main/weather_data.csv`

---

## 5. Пошаговое выполнение задания в Google Colab

### 5.1. Подготовка окружения

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import requests
import io
```

### 5.2. Загрузка данных из репозитория

```python
# Вставьте вашу ссылку
url_data = "https://raw.githubusercontent.com/ваш_логин/weather-forecast-lstm/main/weather_data.csv"

response = requests.get(url_data)
df = pd.read_csv(io.StringIO(response.text), parse_dates=['date'])
print(df.head())
print(df.info())
```

### 5.3. Визуализация и предобработка

Проверим пропуски и построим график.

```python
# Проверка пропусков
print(df.isnull().sum())
# Если есть пропуски, заполним их (например, интерполяцией)
df = df.interpolate()

# Выберем целевую переменную (например, temperature)
target = 'temperature'
values = df[target].values.reshape(-1,1)

# График исходных данных
plt.figure(figsize=(12,4))
plt.plot(df['date'], values)
plt.title(f'Исходный ряд {target}')
plt.xlabel('Дата')
plt.ylabel(target)
plt.show()
```

### 5.4. Масштабирование данных

```python
scaler = MinMaxScaler(feature_range=(0,1))
scaled = scaler.fit_transform(values)

# Сохраним масштабированные значения
print(f"Min: {scaler.data_min_[0]}, Max: {scaler.data_max_[0]}")
```

### 5.5. Создание обучающих последовательностей

Функция `create_sequences` формирует окна заданной длины.

```python
def create_sequences(data, look_back=1):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:i+look_back])
        y.append(data[i+look_back])
    return np.array(X), np.array(y)

look_back = 30  # количество предыдущих дней для прогноза
X, y = create_sequences(scaled, look_back)

print(f"Форма X: {X.shape}, форма y: {y.shape}")
```

### 5.6. Разделение на обучающую и тестовую выборки

Сохраним временной порядок: первые 80% – обучение, последние 20% – тест.

```python
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# LSTM ожидает вход [samples, timesteps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(f"Обучающих примеров: {X_train.shape[0]}, тестовых: {X_test.shape[0]}")
```

### 5.7. Построение LSTM модели

```python
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(look_back, 1)),
    Dropout(0.2),
    LSTM(50),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()
```

### 5.8. Обучение модели

```python
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)
```

### 5.9. Визуализация процесса обучения

```python
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери (MSE)')
plt.show()
```

### 5.10. Прогнозирование на тестовой выборке

```python
y_pred = model.predict(X_test)

# Обратное масштабирование
y_test_orig = scaler.inverse_transform(y_test.reshape(-1,1))
y_pred_orig = scaler.inverse_transform(y_pred)

# Оценка качества
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mae = mean_absolute_error(y_test_orig, y_pred_orig)
print(f"RMSE: {rmse:.2f} °C")
print(f"MAE: {mae:.2f} °C")
```

### 5.11. Визуализация прогноза

```python
plt.figure(figsize=(12,5))
plt.plot(df['date'].iloc[train_size+look_back:], y_test_orig, label='Истинные значения')
plt.plot(df['date'].iloc[train_size+look_back:], y_pred_orig, label='Прогноз', linestyle='--')
plt.legend()
plt.title('Прогноз температуры на тестовой выборке')
plt.xlabel('Дата')
plt.ylabel('Температура')
plt.show()
```

### 5.12. Функция прогнозирования следующих N шагов

```python
def predict_future(model, last_sequence, scaler, n_steps=7):
    """
    Прогнозирует следующие n_steps значений.
    last_sequence: массив последних look_back масштабированных значений.
    """
    current_seq = last_sequence.copy()
    predictions = []
    for _ in range(n_steps):
        input_seq = current_seq.reshape((1, look_back, 1))
        pred = model.predict(input_seq, verbose=0)[0,0]
        predictions.append(pred)
        current_seq = np.append(current_seq[1:], pred)
    return scaler.inverse_transform(np.array(predictions).reshape(-1,1)).flatten()

# Пример: взять последние look_back значений из масштабированного ряда
last_vals = scaled[-look_back:].flatten()
future = predict_future(model, last_vals, scaler, n_steps=7)
print("Прогноз на 7 дней:", future)
```

### 5.13. Сохранение модели и скейлера

```python
model.save('weather_lstm.h5')
with open('scaler.pickle', 'wb') as f:
    pickle.dump(scaler, f)
print("Модель и скейлер сохранены.")
```

### 5.14. Загрузка файлов в репозиторий

1. Скачайте `weather_lstm.h5` и `scaler.pickle` на компьютер.
2. Загрузите их в свой репозиторий.
3. Получите raw‑ссылки (формат `https://github.com/ваш_логин/weather-forecast-lstm/raw/main/weather_lstm.h5` и аналогично для scaler).

### 5.15. Загрузка модели из репозитория и быстрый прогноз

```python
url_model = "https://github.com/ваш_логин/weather-forecast-lstm/raw/main/weather_lstm.h5"
url_scaler = "https://github.com/ваш_логин/weather-forecast-lstm/raw/main/scaler.pickle"

!wget -O weather_lstm.h5 {url_model}
!wget -O scaler.pickle {url_scaler}

loaded_model = load_model('weather_lstm.h5')
with open('scaler.pickle', 'rb') as f:
    loaded_scaler = pickle.load(f)

# Прогноз на 7 дней (нужны последние look_back значений исходного ряда)
# Для примера возьмём последние look_back масштабированных значений из ранее загруженного df
last_vals = scaled[-look_back:].flatten()
future = predict_future(loaded_model, last_vals, loaded_scaler, n_steps=7)
print("Прогноз на 7 дней (загруженная модель):", future)
```

---


## 6. Задания для студентов

### Задание 1. Подготовка репозитория и данных
- Создайте публичный репозиторий на GitHub.
- Выберите реальный метеорологический датасет (или сгенерируйте синтетический) и загрузите его в репозиторий под именем `weather_data.csv`.
- Получите raw‑ссылку на файл.

### Задание 2. Подготовка данных в ноутбуке
- Загрузите данные из репозитория, выведите первые строки, информацию о пропусках.
- Визуализируйте исходный ряд (температуру).
- Выберите целевую переменную и выполните масштабирование.
- Создайте последовательности с `look_back = 30` (объясните, почему выбран этот размер).
- Разделите данные на обучающую (80%) и тестовую (20%) выборки, сохранив порядок.

### Задание 3. Построение модели
- Постройте LSTM‑модель с двумя слоями LSTM и Dropout. Объясните, зачем нужен Dropout.
- Выведите summary.
- Скомпилируйте модель с оптимизатором Adam и функцией потерь MSE.

### Задание 4. Обучение модели
- Обучите модель с EarlyStopping (patience=10).
- Постройте график потерь на обучении и валидации.
- Сохраните обученную модель и скейлер.

### Задание 5. Прогнозирование и оценка
- Получите прогноз на тестовой выборке.
- Вычислите RMSE и MAE. Визуализируйте сравнение реальных и предсказанных значений.
- Проанализируйте, насколько точно модель предсказывает резкие изменения.

### Задание 6. Функция многодневного прогноза
- Реализуйте функцию, которая принимает последние `look_back` наблюдений и возвращает прогноз на следующие N дней.
- Продемонстрируйте её работу (например, прогноз на 7 дней).

### Задание 7. Сохранение и загрузка в репозиторий
- Сохраните модель и скейлер. Загрузите их в репозиторий.
- Получите raw‑ссылки на файлы.

### Задание 8. Загрузка модели из репозитория и быстрый прогноз
- В отдельной ячейке загрузите модель и скейлер из репозитория.
- Выполните прогноз на следующие 7 дней и выведите результат.

### Задание 9. Анализ результатов
- Какие факторы влияют на качество прогноза? Предложите способы улучшения модели (изменение look_back, добавление дополнительных параметров (влажность, давление), использование более глубоких архитектур, включение сезонных признаков).

### Задание 10*. Дополнительно (по желанию)
- Попробуйте использовать двунаправленный LSTM (Bidirectional).
- Добавьте в модель слой `TimeDistributed(Dense(...))` для прогнозирования нескольких шагов одновременно.
- Включите в обучение несколько метеопараметров (многомерный временной ряд).

---

## 7. Заключение

В ходе выполнения этого задания вы:
- создали собственный репозиторий на GitHub и научились загружать туда файлы;
- познакомились с основами анализа временных рядов и применением LSTM для прогнозирования;
- подготовили и масштабировали данные, сформировали обучающие окна;
- построили и обучили LSTM‑модель для прогнозирования температуры;
- оценили качество прогноза с помощью RMSE и визуализации;
- сохранили модель и скейлер, загрузили их из репозитория для повторного использования.

Полученные навыки являются основой для решения более сложных задач прогнозирования в различных областях: финансы, энергетика, транспорт, метеорология.

---